# Load Gold SCD Type 2 Dimensions

Creates and loads `gold.DimHospitals`, `gold.DimDoctors` and `gold.DimPatients` from the silver layer as SCD Type 2 dimensions. Records are upserted on the business key (`<entity>_id`, `updated_at`), surrogate keys start at 5001, and each version carries `EffectiveDate`, `ExpirationDate` and `IsActive`.

In [0]:
dbutils.widgets.text("catalog", "mano_adb_outlook")
catalog = dbutils.widgets.get("catalog")

SK_START = 5001                      # first surrogate key value
HIGH_DATE = "9999-12-31 00:00:00"    # ExpirationDate for the active version

# Silver technical columns that are not carried into the dimension
EXCLUDE_COLS = {"landing_timestamp", "ingestion_timestamp", "DateAdded", "DateUpdated"}

# source silver table -> target gold dimension, surrogate key and business keys
DIM_CONFIG = [
    {"source": "hospitals", "target": "DimHospitals", "sk": "HospitalSK", "id_col": "hospital_id", "version_col": "updated_at"},
    {"source": "doctors",   "target": "DimDoctors",   "sk": "DoctorSK",   "id_col": "doctor_id",   "version_col": "updated_at"},
    {"source": "patients",  "target": "DimPatients",  "sk": "PatientSK",  "id_col": "patient_id",  "version_col": "updated_at"},
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
def load_scd2_dimension(cfg):
    src_tbl = f"{catalog}.silver.{cfg['source']}"
    tgt_tbl = f"{catalog}.gold.{cfg['target']}"
    sk, id_col, ver_col = cfg["sk"], cfg["id_col"], cfg["version_col"]
    keys = [id_col, ver_col]

    # Business attributes taken from silver (technical load columns excluded)
    src_fields = [f for f in spark.table(src_tbl).schema.fields if f.name not in EXCLUDE_COLS]
    attr_cols = [f.name for f in src_fields]
    non_key_cols = [c for c in attr_cols if c not in keys]

    # 1. Create the dimension table if it does not exist
    col_ddl = ",\n  ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in src_fields)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tgt_tbl} (
          {sk} BIGINT NOT NULL,
          {col_ddl},
          EffectiveDate TIMESTAMP,
          ExpirationDate TIMESTAMP,
          IsActive BOOLEAN
        ) USING DELTA
    """)

    sel_cols = ", ".join(f"`{c}`" for c in attr_cols)
    key_join = lambda a, b: " AND ".join(f"{a}.`{k}` = {b}.`{k}`" for k in keys)
    attr_pick = ",\n          ".join(
        f"CASE WHEN s.`{id_col}` IS NOT NULL THEN s.`{c}` ELSE t.`{c}` END AS `{c}`" for c in non_key_cols)

    # 2. Build the staged desired state:
    #    - silver deduplicated on the business key (latest ingestion wins)
    #    - union of all versions (silver + existing gold) per id to derive Expiration/IsActive
    #    - existing versions keep their surrogate key, new versions get max(SK)+n (starting at 5001)
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW stg_{cfg['target']} AS
        WITH src AS (
          SELECT {sel_cols} FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY `{id_col}`, `{ver_col}` ORDER BY ingestion_timestamp DESC) AS rn
            FROM {src_tbl}
            WHERE `{id_col}` IS NOT NULL AND `{ver_col}` IS NOT NULL
          ) WHERE rn = 1
        ),
        versions AS (
          SELECT `{id_col}`, `{ver_col}` FROM src
          UNION
          SELECT `{id_col}`, `{ver_col}` FROM {tgt_tbl}
        ),
        versions_next AS (
          SELECT *, LEAD(`{ver_col}`) OVER (PARTITION BY `{id_col}` ORDER BY `{ver_col}`) AS next_ver
          FROM versions
        ),
        max_sk AS (
          SELECT COALESCE(MAX({sk}), {SK_START - 1}) AS max_sk FROM {tgt_tbl}
        )
        SELECT
          COALESCE(t.{sk},
                   m.max_sk + ROW_NUMBER() OVER (PARTITION BY t.{sk} IS NULL ORDER BY v.`{id_col}`, v.`{ver_col}`)) AS {sk},
          v.`{id_col}`,
          {attr_pick},
          v.`{ver_col}`,
          v.`{ver_col}` AS EffectiveDate,
          COALESCE(v.next_ver, TIMESTAMP'{HIGH_DATE}') AS ExpirationDate,
          v.next_ver IS NULL AS IsActive
        FROM versions_next v
        CROSS JOIN max_sk m
        LEFT JOIN src s ON {key_join('s', 'v')}
        LEFT JOIN {tgt_tbl} t ON {key_join('t', 'v')}
    """)

    # 3. Single atomic MERGE on the business key:
    #    - new business keys are inserted
    #    - existing keys are updated only when attributes or SCD2 columns changed
    #      (e.g. a previous active version gets expired when a newer version arrives)
    compare_cols = non_key_cols + ["EffectiveDate", "ExpirationDate", "IsActive"]
    changed = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in compare_cols)
    update_set = ", ".join(f"t.`{c}` = s.`{c}`" for c in compare_cols)
    all_cols = [sk] + attr_cols + ["EffectiveDate", "ExpirationDate", "IsActive"]
    ins_cols = ", ".join(f"`{c}`" for c in all_cols)
    ins_vals = ", ".join(f"s.`{c}`" for c in all_cols)

    result = spark.sql(f"""
        MERGE INTO {tgt_tbl} t
        USING stg_{cfg['target']} s
        ON {key_join('t', 's')}
        WHEN MATCHED AND ({changed}) THEN UPDATE SET {update_set}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """).collect()[0].asDict()
    print(f"{tgt_tbl}: {result}")
    return result

In [0]:
for cfg in DIM_CONFIG:
    load_scd2_dimension(cfg)

In [0]:
from functools import reduce

checks = []
for cfg in DIM_CONFIG:
    tgt, sk, id_col, ver = f"{catalog}.gold.{cfg['target']}", cfg["sk"], cfg["id_col"], cfg["version_col"]
    checks.append(spark.sql(f"""
        SELECT '{cfg['target']}' AS dimension,
               (SELECT COUNT(*) FROM {catalog}.silver.{cfg['source']}) AS silver_rows,
               COUNT(*) AS dim_rows,
               MIN({sk}) AS min_sk,
               MAX({sk}) AS max_sk,
               COUNT(DISTINCT {sk}) AS distinct_sk,
               COUNT(DISTINCT {id_col}, {ver}) AS distinct_business_keys,
               COUNT(DISTINCT {id_col}) AS distinct_ids,
               SUM(CASE WHEN IsActive THEN 1 ELSE 0 END) AS active_rows,
               SUM(CASE WHEN NOT IsActive AND ExpirationDate = TIMESTAMP'{HIGH_DATE}' THEN 1 ELSE 0 END) AS bad_expired_rows
        FROM {tgt}
    """))

display(reduce(lambda a, b: a.unionByName(b), checks))